# Tool을 사용하는 챗봇 구현


## OpenAI LLM 준비


In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# .env 파일 로드 및 API 키 확인
load_dotenv()

# OpenAI LLM 준비
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)

## 데코레이터를 통한 Tool 구성


In [ ]:
from langchain.tools import tool
from langchain_teddynote.tools import GoogleNews

google_news = GoogleNews()


@tool
def news_search(query: str) -> str:
    """뉴스 검색 도구를 호출해 결과를 반환한다."""
    results = google_news.search_by_keyword(query, k=5)
    text = (
        "\n".join(
            f"- {r.get('content','(내용없음)')} | {r.get('url','')}" for r in results
        )
        or "검색 결과 없음"
    )
    return text

## Agent 생성


In [ ]:
from langchain.agents import create_agent

tools = [news_search]

agent = create_agent(
    model=llm,
    tools=tools,
)

## 프롬프트 메시지 구성


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 명탐정 코난 영화 전문가입니다. 뉴스 검색 도구를 호출해 질문에 답변하세요.",
        ),
        (
            "human",
            "{question} **뉴스 검색 도구의 결과를 바탕으로 답변해주세요.**",
        ),
    ]
)
prompt_messages = prompt.format_messages(
    question="뉴스 검색 도구를 통해 가장 최근에 개봉한 명탐정 코난 영화는 무엇인지 요약해주세요."
)

In [ ]:
response = agent.invoke({"messages": prompt_messages})

print("에이전트 응답:", response["messages"][-1].content)
response["messages"]